# EMBVNVD207CC6

Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
import math
import os
import gc
from pathlib import Path
import re
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Set directory to project root
def find_project_root(start: Path = Path().absolute()) -> Path:
    for parent in start.parents:
        if (parent / "requirements.txt").exists(): return parent
    return start 
os.chdir(find_project_root())

# Custom packages
from tools.filter import FilterDF as fdf
from tools.benchmarks import ParetoAnalysis as pa
from tools.benchmarks import AccuracyCalculation as ac
from tools.integrity_fixes import DataFixer as fix, DataExporter as exporter
from tools.coverage_functions import plot_time_series
from tools.labeling_functions import plot_dish_time_series

# Preemptively set new Pandas option, also set matplotlib to close
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

# Allow reloading of custom Python classes without resetting kernel
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Read data from parquet files

In [ ]:
# Load formatted data
%store -r static_data_merged
%store -r sales_data_merged

# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"data/2_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    
    %store static_data_merged
    
# Data already exists
else:
    static_data = static_data_merged.copy()

# 

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"data/2_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
        
    %store sales_data_merged
    
# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()

# 

# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

# True promos
%store -r before_after_details_true
if 'before_after_details_true' not in locals():
    before_after_details_true = pd.read_csv('data/3_data_parquet_relabeled/before_after_details_true.csv', index_col='location_id')
    %store before_after_details_true

# Timezones
%store -r timezones
if 'timezones' not in locals():
    timezones = pd.read_csv('data/3_data_parquet_relabeled/timezones.csv', index_col='location_id')['timezone'].to_dict()
    for loc_id, df in sales_and_menu_data.items():
        df.index = df.index.tz_convert(timezones[loc_id])
        sales_and_menu_data[loc_id] = df
    %store timezones

%store -r restaurants_by_4m_coverage
if 'restaurants_by_4m_coverage' not in locals():
    restaurants_by_4m_coverage = pd.read_csv('data/3_data_parquet_relabeled/restaurants_by_4m_coverage.csv')['location_id'].tolist()
    %store restaurants_by_4m_coverage

loc_id = 'EMBVNVD207CC6'
df_uncleaned = sales_and_menu_data[loc_id]

locations = list(sales_and_menu_data.keys())
for other_loc_id in locations:
    if other_loc_id != loc_id:
        del sales_and_menu_data[other_loc_id]
        del sales_data_merged[other_loc_id]
gc.collect()

In [ ]:
# df = df.assign(item_modifications = lambda df: df['item_modifications'].str.replace("Un'Chicken|Un'chicken", 'Unchicken', regex=True))

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
df_uncleaned.query('item_name.str.contains("Vegan") or item_modifications.str.contains("Vegan")')['item_quantity'].sum()

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
df_uncleaned.loc[promo_date:promo_date+pd.DateOffset(days=60)].query('item_name.str.contains("Vegan") or item_modifications.str.contains("Vegan")')['item_quantity'].sum()

In [ ]:
df_uncleaned.query('dish_category != "Alcohol"')['item_name'].value_counts()

In [ ]:
df_uncleaned.query('item_name.str.contains("Vegan") or item_modifications.str.contains("Vegan")').head(5)

In [ ]:
plot_time_series('EMBVNVD207CC6', 
                 df_uncleaned.query('item_name.str.contains("Vegan") or item_modifications.str.contains("Vegan")'), 
                 before_after_details_true, 
                 freq='7D', 
                 subset=False)
plt.show()

In [ ]:
plot_time_series('EMBVNVD207CC6', 
                 df_uncleaned.query('item_name.str.contains("Vegan") or item_modifications.str.contains("Vegan")'), 
                 before_after_details_true, 
                 freq='D', 
                 subset=True)
plt.show()

# Start

Important note, this restaurant has "Unchicken" as a modification

In [ ]:
food_labeled_unsure = ['3 Cheese Caramelized Onion Dip',
                       'Standard',
                       'Snacks',
                       'Nonspecific Food Mod',
                       'Vegan',
                       'Green Chile Cheese Frittata',
                       'Brownie',
                       'Blueberries',
                       'Apple Crumble']

non_food_categories = ["Coffee & Tea", 
                       "Water",
                       "Smoothie",
                       "Soda",
                       "Merch",
                       "Drink",
                       "Alcohol"]

odd_named_alcohol = ["Blueberrye",
                     "Green Giant",
                     "Green Giant Cans",
                     "B.A. Taster"]

food_df = (df
           .query('(dish_category != "Unsure") or item_name.isin(@food_labeled_unsure)')
           .query('~item_name.isin(@odd_named_alcohol)')
           .query('~item_name.str.contains(r"\\d", regex=True)')
           .query('~dish_category.isin(@non_food_categories)')
           #.query('is_plant_based == "Yes"')
           #[['item_name',
             #'dish_category',
             #'is_plant_based'
             #]]
             )

food_items = list(food_df['item_name'].unique())

In [ ]:
plot_dish_time_series(food_df.query('item_name.str.contains("Vegan") or item_modifications.str.contains("Vegan")'), loc_id, before_after_details_true)

# Type the entries with modifications here:

In [ ]:
# Original item name, animal-based ingredient, new name for item with animal-based ingredient , custom/
# 0 or 1, 1 if their modifications were within what was orderable on their website
modification_name_changes = [
    ('Soft Pretzel And Big Hop Beer Mustard (On-Site Order)','Cheese','Pretzel With Cheese'),
    ('Breadsticks','Vegan|Vegan Cheese|No Cheese','Vegan Breadsticks'),
    ('Pepperoni Pizza (On-Site Order)','All Vegan','Vegan Pepperoni Pizza'),
    ('Pepperoni Pizza (On-Site Order)','Vegan Pepperoni|','Vegetarian Pepperoni Pizza'),
    ('Cheese Pizza (On-Site Order)','Vegan','Vegan Cheese Pizza'),
    ('Pepperoni Pizza','Vegetarian','Vegetarian Pepperoni Pizza'),
    ('Pepperoni Pizza','Vegan Pie','Vegan Pepperoni Pizza'),
    ('Cheese Pizza','Vegan Cheese','Vegan Cheese Pizza'),
    ('Totchos','No Slaw|No Cole Slaw|No Cheese','Vegan Totchos'),
    ('Cheese Pizza Slice','Vegan|No Cheese','Vegan Cheese Pizza Slice'),
    ('Ranch Side','Vegan','Vegan Ranch Side'),
    ('Green Chile Cheeseburger','No Cheese','Vegan Green Chile Cheeseburger'),
    ("Take N' Bake Pepperoni Pizza","Vegan","Vegan Take N' Bake Pepperoni Pizza"),
    ("Take N' Bake Pepperoni Pizza","Vegetarian","Vegetarian Take N' Bake Pepperoni Pizza"),
    ("Take 'N Bake Cheese Pizza","Vegan","Vegan Take 'N Bake Cheese Pizza"),
    ("Chicken Salad","Sub Cauli","Cauli Salad"),
    ("Cheese Quesadilla With Salsa","Bean","Bean Cheese Quesadilla With Salsa"),
    ("Gratitude Pretzel","Beer Cheese","Cheese Gratitude Pretzel"),
    ("Roasted Cauliflower Poboy","Vegan","Vegan Roasted Cauliflower Poboy"),
    ("Roasted Cauliflower Poboy","Shrimp","Shrimp Roasted Cauliflower Poboy"),
    ("Caprese Salad","Vegan","Vegan Caprese Salad")
    ]

# List the items here

In [ ]:
# Is meat but was labelled plant-based
meat = ['Shrimp Roasted Cauliflower Poboy'
        ]

# Is vegetarian but was labelled plant-based or meat
vegetarian = ['Pretzel With Cheese',
              'Breadsticks',
              'Vegetarian Pepperoni Pizza',
              'Cheese Pizza (On-Site Order)',
              'Cheese Pizza',
              'Pretzels',
              'Totchos',
              'Cheese Pizza Slice',
              'Ranch Side',
              'Cheese Board',
              'Cheese Plate',
              'Green Chile Cheeseburger',
              'Margarita Pizza',
              "Vegetarian Take N' Bake Pepperoni Pizza",
              'Green Chile Cheese Frittata',
              'Buffalo Broccoli Dip',
              'White Pizza',
              "Take 'N Bake Cheese Pizza",
              "Mac & Cheese",
              "Cauli Salad",
              "Bean Cheese Quesadilla With Salsa",
              "Coleslaw",
              "Cheese Gratitude Pretzel",
              "Roasted Cauliflower Poboy",
              "Caprese Salad",
              "Cheesey Tots",
              "Brownie",
              "Elote Salad",
              "Catering Breadstix",
              "Cheese Tostada With Salsa",
              "Catering Pretzel Bites",
              "Apple Crumble",
              "Potater Chipz"
              ]

# Is vegan but was labelled meat
vegan = ['Soft Pretzel And Big Hop Beer Mustard (On-Site Order)',
         'Vegan Breadsticks',
         'Vegan Pepperoni Pizza',
         'Vegan Cheese Pizza',
         'Vegan Totchos',
         'Vegan Cheese Pizza Slice',
         'Vegan Ranch Side',
         'Soft Pretzel And Big Hop Beer Mustard',
         'Vegan Green Chile Cheeseburger',
         "Vegan Take N' Bake Pepperoni Pizza",
         "Vegan Take 'N Bake Cheese Pizza",
         "Gratitude Pretzel",
         "Vegan Roasted Cauliflower Poboy",
         "Vegan Caprese Salad",
         "Chips"]

# If they are not vegetarian or vegan, they can default be labeled as animal-based, so the third list is unnecessary



# Supplemental labels, unnecessary for now

non_alcoholic_drinks = ['',
                        '',
                        '',
                        '',]

alcoholic_drinks = []

merch = ['Blackstrap Cake Mix',
         "Eliza'S Lime Curd!",
         '',]

rare = ['',
        '',
        '',]

unknown = ['Pizza Boy Bottle',
           '',
           '',]

items_to_remove = ['',
                   '',
                   '',
                   '',] # nonfood